# Grand Capstone Project

In this **Grand Capstone Project**, we will tie all these independent skills together to build a complete, professional, end-to-end data pipeline. We will analyze real-world tech job market data—specifically focusing on **Data Analyst, Data Scientist, and Data Engineer** job postings. You will learn how to transition from loading messy raw data all the way to extracting high-value, clean insights ready for business dashboards.


## Setting Up an End-to-End Data Pipeline

### 1. What You Will Learn and Why It Matters
In the real world, data analysis is rarely a single-step operation. You do not just run a groupby and call it a day. Instead, professional data engineers and analysts build **data pipelines**. A pipeline is a sequential series of steps where the output of one step becomes the input of the next:

$$\text{messy raw data} \longrightarrow \text{Loading} \longrightarrow \text{Cleaning} \longrightarrow \text{Transformation} \longrightarrow \text{Aggregation} \longrightarrow \text{Visual-Ready Insights}$$

Understanding how to structure this pipeline programmatically in Pandas keeps your code readable, reproducible, and highly optimized for production environments.


### Conceptual Explanation & Real-World Analogy
Think of a data pipeline like a **professional water filtration facility**:
1.  **Water Ingestion (Loading)**: We draw raw, muddy river water into our system.
2.  **Filtration & Sediment Removal (Cleaning)**: We filter out dirt, debris, and twigs (removing `NaN` values, filtering out irrelevant rows, dropping duplicates).
3.  **Chemical Balancing (Transformation)**: We add minerals and adjust the pH to make it safe (standardizing dates, parsing text columns, transforming string types).
4.  **Distribution Prep (Aggregation)**: We bottle the filtered water into standardized cases for different regions (grouping, reshaping, and summarizing).

If you skip the cleaning phase and jump straight to bottling, you end up distributing muddy water to your customers. Similarly, running aggregations on uncleaned datasets will produce highly inaccurate business insights.

### 3. Code Walkthrough: Building the Pipeline

Let's build a functional 5-stage pipeline to analyze real-world tech job postings. We will use a dictionary-based dataset representing job postings scraped from various boards.



In [11]:
import pandas as pd
import numpy as np

# --- STAGE 1: RAW DATA INGESTION ---
raw_data = {
    'job_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'title': ['Data Analyst', 'Senior Data Scientist', 'Data Engineer', 'Data Analyst', 'Data Scientist', 'Data Engineer', 'Director of Data Science', 'Data Analyst'],
    'company': ['TechCorp', 'InnoSoft', 'CloudNet', 'TechCorp', 'InnoSoft', 'CloudNet', 'InnoSoft', 'AlphaData'],
    'skills_required': ['Python, SQL, Excel', 'Python, R, SQL, PyTorch', 'SQL, Python, Spark, AWS', 'SQL, Excel, Tableau', 'Python, SQL, PyTorch', 'SQL, Spark, AWS', 'Python, SQL, PyTorch', 'Python, SQL, Tableau'],
    'salary_usd': [85000, 145000, 110000, np.nan, 120000, np.nan, 195000, 90000],
    'posted_date': ['2023-01-15', '2023-01-20', '2023-02-11', '2023-02-15', '2023-03-05', '2023-03-10', '2023-03-15', '2023-03-22']
}

df_raw = pd.DataFrame(raw_data)
print("--- Stage 1: Raw Data Ingested ---")
print(f"Shape: {df_raw.shape}")

--- Stage 1: Raw Data Ingested ---
Shape: (8, 6)



####  Clean the Data
We need to:
1. Standardize string column headers and lowercase titles for matching .
2. Identify and handle missing salaries using `.dropna()`.
3. Convert date strings into standardized datetime objects.


In [12]:
# --- STAGE 2: CLEANING & STANDARDIZATION ---
df_cleaned = df_raw.copy()

# 1. Lowercase titles and strip whitespace to ensure perfect grouping
df_cleaned['title'] = df_cleaned['title'].str.strip().str.title() # e.g. "senior data scientist" -> "Senior Data Scientist"

# 2. Drop rows with missing salaries (we only want verified salary analytics)
df_cleaned.dropna(subset=['salary_usd'], inplace=True)

# 3. Convert posted_date to datetime64
df_cleaned['posted_date'] = pd.to_datetime(df_cleaned['posted_date'])

print("--- Stage 2: Cleaned and Standardized ---")
print(df_cleaned[['title', 'salary_usd', 'posted_date']])

--- Stage 2: Cleaned and Standardized ---
                      title  salary_usd posted_date
0              Data Analyst     85000.0  2023-01-15
1     Senior Data Scientist    145000.0  2023-01-20
2             Data Engineer    110000.0  2023-02-11
4            Data Scientist    120000.0  2023-03-05
6  Director Of Data Science    195000.0  2023-03-15
7              Data Analyst     90000.0  2023-03-22


#### Data Transformation
We need to:
1. Extract the month number from the date to track monthly trends.
2. Explode the comma-separated `skills_required` column into individual skill rows.


In [13]:
# --- STAGE 3: TRANSFORMATION & EXPLODING ---
df_transformed = df_cleaned.copy()

# Extract month
df_transformed['posted_month'] = df_transformed['posted_date'].dt.month

# Convert comma-separated string to a list of skills
df_transformed['skills_list'] = df_transformed['skills_required'].str.split(', ')

# Explode the list column
df_exploded = df_transformed.explode('skills_list')

print("--- Stage 3: Exploded Skills ---")
print(df_exploded[['title', 'skills_list', 'posted_month', 'salary_usd']].head(10))

--- Stage 3: Exploded Skills ---
                   title skills_list  posted_month  salary_usd
0           Data Analyst      Python             1     85000.0
0           Data Analyst         SQL             1     85000.0
0           Data Analyst       Excel             1     85000.0
1  Senior Data Scientist      Python             1    145000.0
1  Senior Data Scientist           R             1    145000.0
1  Senior Data Scientist         SQL             1    145000.0
1  Senior Data Scientist     PyTorch             1    145000.0
2          Data Engineer         SQL             2    110000.0
2          Data Engineer      Python             2    110000.0
2          Data Engineer       Spark             2    110000.0


#### Multi-Level Aggregation & Reshaping
We will group by both `skills_list` and `title` to find the median salary and count of postings for each skill-job combination.

In [14]:
# --- STAGE 4: AGGREGATION & RESHAPING ---
# Group by Skill and Title to find count of postings and median salary
pipeline_metrics = df_exploded.groupby(['skills_list', 'title']).agg(
    postings_count=('job_id', 'count'),
    median_salary=('salary_usd', 'median')
).reset_index()

# Pivot the table to show Median Salary for each Skill across different Job Titles
pivot_salary = pipeline_metrics.pivot(
    index='skills_list',
    columns='title',
    values='median_salary'
).fillna(0) # Fill unrepresented combinations with 0 for clean display

print("--- Stage 4: Reshaped Skill Salary Matrix ---")
print(pivot_salary)

--- Stage 4: Reshaped Skill Salary Matrix ---
title        Data Analyst  Data Engineer  Data Scientist  \
skills_list                                                
AWS                   0.0       110000.0             0.0   
Excel             85000.0            0.0             0.0   
PyTorch               0.0            0.0        120000.0   
Python            87500.0       110000.0        120000.0   
R                     0.0            0.0             0.0   
SQL               87500.0       110000.0        120000.0   
Spark                 0.0       110000.0             0.0   
Tableau           90000.0            0.0             0.0   

title        Director Of Data Science  Senior Data Scientist  
skills_list                                                   
AWS                               0.0                    0.0  
Excel                             0.0                    0.0  
PyTorch                      195000.0               145000.0  
Python                       195000.0 


### Common Pitfalls for Beginners
*   **The Inplace-Chaining Trap**: Trying to chain `.dropna()` with `.explode()` in a single unreadable line. Always separate your logical stages so you can run intermediate `.info()` or `.shape` checks.
*   **Forgetting to Reset Index**: After exploding your list, your row indices will duplicate (e.g., index `0` will appear three times if the row had three skills). This can cause massive issues if you merge the DataFrame later. Always use `.reset_index(drop=True)` after exploding.


## The Grand Capstone - Tech Job Stack Analytics

Now, let's step up to a fully realized capstone challenge.

### The Scenario
You have been hired by a leading global recruitment agency. They have scraped a raw dataset containing thousands of postings for data roles . The agency wants you to build a robust Python program that takes this messy file and outputs a clean, visual-ready **Skill-to-Salary Demand Matrix** that compares the market value of **Python, SQL, Excel, and PyTorch** across different job titles.

### The Comprehensive Python Script
Create a new file named `tech_stack_analytics.py` in your local directory and run the following code . (This code loads and cleans a larger, simulated raw dataset of tech postings) :


In [15]:
import pandas as pd
import numpy as np

def run_tech_stack_pipeline():
    # 1. Ingestion: Simulating 1,000 messy scraped rows
    np.random.seed(42)
    n_records = 1000

    titles = ['Data Analyst', 'Data Scientist', 'Data Engineer', 'Machine Learning Engineer', 'Software Engineer']
    companies = ['FinTech Group', 'HealthAI', 'Global Logistics', 'RetailCorp', 'MegaSocial']
    skills_combos = [
        'Python, SQL, Excel',
        'Python, SQL, Tableau, PowerBI',
        'SQL, Spark, Python, AWS',
        'Python, PyTorch, SQL',
        'Java, C++, Python',
        'Excel, SQL, PowerPoint'
    ]

    raw_df = pd.DataFrame({
        'job_id': range(1000, 1000 + n_records),
        'job_title': np.random.choice(titles, size=n_records, p=[0.3, 0.25, 0.2, 0.15, 0.1]),
        'company_name': np.random.choice(companies, size=n_records),
        'skills': np.random.choice(skills_combos, size=n_records),
        'salary_usd': np.random.normal(loc=115000, scale=35000, size=n_records),
        'posted_date': pd.date_range(start='2023-01-01', periods=n_records, freq='h').strftime('%Y-%m-%d %H:%M:%S')
    })

    # Intentionally injecting missing values and inconsistent formatting to mimic real scraping
    raw_df.loc[raw_df['salary_usd'] < 60000, 'salary_usd'] = np.nan
    raw_df.loc[np.random.choice(raw_df.index, 50), 'job_title'] = '  data analyst  '
    raw_df.loc[np.random.choice(raw_df.index, 50), 'job_title'] = 'DATA SCIENTIST'

    print("--- INGESTION COMPLETE ---")
    print(f"Scraped Raw Shape: {raw_df.shape}")
    print(raw_df[['job_title', 'skills', 'salary_usd']].head(3))

    # 2. Cleaning & Standardization
    print("--- STARTING DATA CLEANING STAGE ---")
    df_clean = raw_df.copy()

    # Standardize job title strings
    df_clean['job_title'] = df_clean['job_title'].str.strip().str.title()

    # Filter for our core data roles
    core_roles = ['Data Analyst', 'Data Scientist', 'Data Engineer']
    df_clean = df_clean[df_clean['job_title'].isin(core_roles)]

    # Handle missing salaries by replacing them with the median salary for that job title
    # We use a group-wise transformation!
    df_clean['salary_usd'] = df_clean.groupby('job_title')['salary_usd'].transform(lambda x: x.fillna(x.median()))

    # Convert posted_date to Datetime format
    df_clean['posted_date'] = pd.to_datetime(df_clean['posted_date'])

    # 3. Transformation & Exploding
    print("--- STARTING TRANSFORMATION STAGE ---")
    df_clean['skills_split'] = df_clean['skills'].str.split(', ')
    df_exploded = df_clean.explode('skills_split')
    df_exploded.reset_index(drop=True, inplace=True)

    # Target only our core analytical skills
    target_skills = ['Python', 'SQL', 'Excel', 'PyTorch', 'Tableau']
    df_exploded = df_exploded[df_exploded['skills_split'].isin(target_skills)]

    # 4. Multi-Level Aggregation
    print("--- STARTING AGGREGATION STAGE ---")
    # Find count of postings, median salary, and mean salary for each skill-job combination
    aggregated = df_exploded.groupby(['skills_split', 'job_title']).agg(
        postings_count=('job_id', 'count'),
        median_salary=('salary_usd', 'median')
    ).reset_index()

    # 5. Pivoting for Business Reporting
    print("--- GENERATING REPORTING MATRIX ---")
    # Pivot so columns are Job Titles and index are Skills
    final_report = aggregated.pivot(
        index='skills_split',
        columns='job_title',
        values='median_salary'
    ).round(2)

    # Sort final report based on Data Scientist median salaries descending
    final_report = final_report.sort_values(by='Data Scientist', ascending=False)

    print("=== THE RECRUITMENT MARKET VALUE MATRIX ===")
    print(final_report)

run_tech_stack_pipeline()

--- INGESTION COMPLETE ---
Scraped Raw Shape: (1000, 6)
           job_title                         skills    salary_usd
0     Data Scientist  Python, SQL, Tableau, PowerBI  107329.53931
1  Software Engineer        SQL, Spark, Python, AWS  157970.23070
2      Data Engineer         Excel, SQL, PowerPoint   77780.92977
--- STARTING DATA CLEANING STAGE ---
--- STARTING TRANSFORMATION STAGE ---
--- STARTING AGGREGATION STAGE ---
--- GENERATING REPORTING MATRIX ---
=== THE RECRUITMENT MARKET VALUE MATRIX ===
job_title     Data Analyst  Data Engineer  Data Scientist
skills_split                                             
Tableau          113559.07      116336.22       119066.06
Python           115501.62      116336.22       117711.58
PyTorch          115809.86      122420.31       117711.58
SQL              115501.62      116336.22       117711.58
Excel            118963.86      109501.55       117392.97


### Step-by-Step Breakdown of the Logic
1.  **Group-wise Imputation (`.transform`)**: Instead of deleting missing values, we filled `NaN` salaries with the **median salary** of that specific job title using `groupby().transform()`. This is an advanced machine learning data prep standard.
2.  **.str.split() and .explode()**: We parsed the comma-separated strings into individual Pandas lists, then exploded them to count skills independently.
3.  **Filtered Pivot**: We pivoted the exploded combinations into a dense matrix comparing salaries, showing exactly which skills are the highest-paying across key roles .


## Common Pitfalls in Large-Scale Data Visualizations

When dealing with massive files (e.g., hundreds of megabytes or gigabytes of log files), writing naive Pandas scripts can easily exhaust system RAM or introduce subtle logic bugs.

### Memory Leaks & Memory Preservation
Every time you load a massive DataFrame or perform a copy operation, you occupy RAM [875]. If you are generating hundreds of reports in a loop, memory can accumulate rapidly [418]:
*   **The Problem**: Modifying DataFrames without releasing older, unneeded variable references.
*   **The Fix**: Use `del df_temp` followed by importing and invoking Python’s garbage collector:
    ```python
    import gc
    del df_raw  # Free up reference
    gc.collect()  # Forces system to clear unused memory blocks
    ```
*Additionally, always release plotting figures. If you generate 1,000 matplotlib charts in a loop without writing `plt.close()`, your program will run out of memory and crash.*


### Slices vs. Copies

When you filter a table, you get back a **slice** — a smaller piece of the big table. The old question was: if I write into that slice, does the **original** table change too?

For years Pandas could not answer this reliably, so it printed a famous message called `SettingWithCopyWarning`.

**This changed in Pandas 3.0** (the version in this project). Pandas 3 uses a rule called **Copy-on-Write**: writing into a slice *never* touches the original table, and there is no warning any more. The answer is now always "no".

So the example below no longer warns. Run it and look at the two tables it prints:



In [16]:
import pandas as pd

df = pd.DataFrame({
    'job_title': ['Data Analyst', 'Data Scientist', 'Data Analyst', 'Data Engineer'],
    'salary_usd': [85000.0, 120000.0, 90000.0, 110000.0]
})

# Take a slice: only the Data Analyst rows
df_analysts = df[df['job_title'] == 'Data Analyst']

# Now write into that slice
df_analysts['salary_usd'] = df_analysts['salary_usd'] + 5000

print("--- the slice: salaries went UP by 5000 ---")
print(df_analysts)

print()  # blank line between the two tables
print("--- the original df: completely UNCHANGED ---")
print(df)


--- the slice: salaries went UP by 5000 ---
      job_title  salary_usd
0  Data Analyst     90000.0
2  Data Analyst     95000.0

--- the original df: completely UNCHANGED ---
        job_title  salary_usd
0    Data Analyst     85000.0
1  Data Scientist    120000.0
2    Data Analyst     90000.0
3   Data Engineer    110000.0


*   **What happened?**: `df_analysts` got its own private copy of the numbers. The original `df` was not touched. In Pandas 2 and older this same code printed `SettingWithCopyWarning`, because back then the original *might* have changed.
*   **Still the best habit**: write `.copy()` yourself, right after the slice. It costs nothing extra here, it says clearly "I want my own separate table", and it keeps your code working on older Pandas versions too:


In [17]:
# .copy() says: "give me my own separate table"
df_analysts = df[df['job_title'] == 'Data Analyst'].copy()
df_analysts['salary_usd'] = df_analysts['salary_usd'] + 5000  # 100% Safe!

print(df_analysts)


      job_title  salary_usd
0  Data Analyst     90000.0
2  Data Analyst     95000.0
